### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [547]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [548]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

### Data generator

##### Support functions

In [549]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [550]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.6, 1.5), (0.6, 0.65, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [551]:
############################### Function to generate the credit to be requested ##################################
# Note the credits will be exactly for one year
def generate_credit_requested(income): ### It will depend on the income
    # The will maximum enter as for a credit that represent between 50% and 150% of their income and the probabilities of any values are equal, so a uniform distribution
    percentage_of_monthly_income = random.uniform(0.5, 1.5)
    yearly_income = income * 12 # must be changes once dynamically made
    credit_requested_yearly = yearly_income *percentage_of_monthly_income
    credit_requested_monthly = credit_requested_yearly / 12
    percentage_credit_month_income = credit_requested_monthly/income
    return credit_requested_monthly, percentage_credit_month_income
    

In [552]:
############################### Function to generate the whether the person defaults or not ##################################
def generate_default_label(profession, past_credits, debt_to_income_ratio_before_credit, credit_to_income_ratio):
    """This simulates a default label (0/1) based on financial risk factors."""
    """What we will use will be the """
    
    # Configurable risk settings per profession
    risk_settings = {
        "Unemployed_LowSkilled":     {"base": 0.15, "weights": (0.30, 0.6, 0.4)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "LowSkilled":                {"base": 0.08, "weights": (0.20, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_MediumSkilled": {"base": 0.12, "weights": (0.30, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "MediumSkilled":            {"base": 0.05, "weights": (0.20, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_HighSkilled":   {"base": 0.09, "weights": (0.30, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "HighSkilled":              {"base": 0.2,  "weights": (0.20, 0.4, 0.2)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
    }

    # setting variable is sett to retrieve the element of reisk_settings for the profession given
    settings = risk_settings.get(profession)
    # In case there is the profession given for the function does not match any of the professions above listed
    if not settings:
        raise ValueError(f"Unknown profession: {profession}")
    
    # Defining the weigths for the calcualtion of the probability of default
    w1, w2, w3 = settings["weights"]
    # Defining the base probability of default
    base = settings["base"]

    # Calculating the risk factor with the weigths
    risk_factor = past_credits * w1 + debt_to_income_ratio_before_credit * w2 + credit_to_income_ratio * w3
    # Defining the default probability
    default_probability = min(1, base + risk_factor)

    # We want a non-deterministic y-categorical variable that will make that same profiles will not always lead to the same result
    # So even if two people may fall on the same profile, maybe they will not default
    # return 1 if random.random() < default_probability else 0
    return int(random.random() < default_probability)

##### Data Generator for the original state of individuals

In [553]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        
        # Debt to income ratio: PARTIAL, before the credit
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio_partial = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio_partial = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        monthly_credit = generate_credit_requested(monthly_income)[0]
        credit_to_income_ratio = generate_credit_requested(monthly_income)[1]
        
        # calculating the monthly debt if the monthluy credit is issued
        debt_after_credit = savings_debt - monthly_credit
        if debt_after_credit > 0: # if still the monthly savings are higher than the credit, then the total debt to incom ratio should be zero
            debt_to_income_ratio_total = 0
        else: 
            debt_to_income_ratio_total = abs(debt_after_credit/monthly_income)
        
        # ---------------- Y-Variable --------------------------------#
        default_not_default = generate_default_label(profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio)
        
        data.append({
            'name': name, # independent
            'age': age, # independent
            'educational level': education_level, # independent
            'number of not paid past credits': past_credits, # independent
            'dependents': dependents, # independent
            'profession': profession, # depends on education
            'monthly income': monthly_income, # depends on profession and age
            'monthly expenditure': monthly_expenditure, # depends on income, mean income per age and profession, and the number of dependents
            'savings (debt)': savings_debt, # monthly income - monthly expenditure
            'debt-to-income ratio before credit': debt_to_income_ratio_partial, # abs(savings_debt/monthly_income)
            'credit: monthly amount': monthly_credit, # depends on the income
            'credit-to-income ratio': credit_to_income_ratio, # credit/income
            'debt-to-income ratio after credit': debt_to_income_ratio_total, # abs((savings_debt - credit)/monthly_income)
            'y-categorical-default': default_not_default # depending on profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [554]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,debt-to-income ratio after credit,y-categorical-default
0,name0,38,high school or lower,0,1,LowSkilled,952.0,1440.368618,-488.368618,0.512992,1330.095876,0.687062,1.910152,0
1,name1,32,post graduate degree,0,1,HighSkilled,3092.0,2465.972725,626.027275,0.000000,1851.645292,0.535636,0.396384,1
2,name2,34,high school or lower,0,1,LowSkilled,1343.0,657.893623,685.106377,0.000000,1636.794542,0.797688,0.708629,0
3,name3,60,bachelor degree,1,0,HighSkilled,6376.0,6100.486048,275.513952,0.000000,6392.332779,0.900734,0.959351,0
4,name4,57,high school or lower,0,0,LowSkilled,2463.0,2171.529759,291.470241,0.000000,3136.363298,1.392856,1.155052,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,48,bachelor degree,1,0,HighSkilled,9316.0,6361.661732,2954.338268,0.000000,4960.434197,1.006062,0.215339,1
996,name996,40,ausbildung,0,0,MediumSkilled,2624.0,1089.893142,1534.106858,0.000000,3299.763186,0.852561,0.672887,1
997,name997,43,high school or lower,2,1,LowSkilled,1461.0,1615.828788,-154.828788,0.105975,2022.859947,0.672195,1.490547,1
998,name998,41,ausbildung,1,1,MediumSkilled,1953.0,1816.165055,136.834945,0.000000,2228.939831,0.733985,1.071226,0


##### DF statistics

In [555]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,debt-to-income ratio after credit,y-categorical-default
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.044000,0.496000,0.577000,3648.058000,3306.092801,341.965199,0.101939,3675.744261,0.990784,0.937731,0.515000
std,9.071955,0.714489,0.834726,2636.143988,2515.333988,1647.329788,0.214879,2979.332423,0.291225,0.432117,0.500025
min,30.000000,0.000000,0.000000,500.000000,367.042918,-11402.997690,0.000000,253.790785,0.501339,0.000000,0.000000
25%,37.000000,0.000000,0.000000,1778.000000,1480.389321,-260.749735,0.000000,1629.374924,0.733544,0.641622,0.000000
50%,45.000000,0.000000,0.000000,2754.500000,2426.895892,250.457183,0.000000,2644.537244,0.982873,0.914216,1.000000
75%,53.000000,1.000000,1.000000,4867.750000,4508.737974,869.153584,0.113841,4829.708868,1.239299,1.185821,1.000000
max,60.000000,3.000000,4.000000,17100.000000,18648.997690,10007.291505,1.605876,19354.144168,1.499588,3.001871,1.000000


Monthly income by profession

In [556]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,371.0,6133.471698,2651.038960,1828.0,4154.0,5806.0,7626.0,17100.0
LowSkilled,279.0,1640.949821,542.774306,517.0,1192.0,1587.0,2036.5,3246.0
MediumSkilled,255.0,2897.901961,1015.687232,1185.0,2142.0,2683.0,3529.5,6328.0
Unemployed_HighSkilled,37.0,2959.459459,1109.481726,1500.0,2250.0,3000.0,4000.0,4500.0
Unemployed_LowSkilled,31.0,803.225806,197.878533,500.0,600.0,900.0,950.0,1100.0
Unemployed_MediumSkilled,27.0,1531.481481,388.326822,900.0,1225.0,1500.0,2000.0,2000.0


Debt-to-income ratio by profession

In [557]:
df_1.groupby('profession')['debt-to-income ratio before credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,371.0,0.128506,0.252602,0.0,0.0,0.0,0.156021,1.573696
LowSkilled,279.0,0.076893,0.171645,0.0,0.0,0.0,0.068058,1.016119
MediumSkilled,255.0,0.103855,0.222523,0.0,0.0,0.0,0.116810,1.605876
Unemployed_HighSkilled,37.0,0.062700,0.120452,0.0,0.0,0.0,0.036672,0.448407
Unemployed_LowSkilled,31.0,0.080116,0.119783,0.0,0.0,0.0,0.119021,0.426362
Unemployed_MediumSkilled,27.0,0.056434,0.094455,0.0,0.0,0.0,0.092257,0.344630


In [558]:
df_1.groupby('profession')['debt-to-income ratio after credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,371.0,0.971506,0.460695,0.000000,0.660963,0.953525,1.220115,2.586503
LowSkilled,279.0,0.907669,0.393093,0.000000,0.621574,0.888051,1.164982,2.306777
MediumSkilled,255.0,0.929017,0.457941,0.000000,0.635470,0.895800,1.158517,3.001871
Unemployed_HighSkilled,37.0,0.906507,0.347793,0.269353,0.704506,0.864976,1.143515,1.734948
Unemployed_LowSkilled,31.0,0.898397,0.348436,0.318375,0.631264,0.942440,1.090735,1.529592
Unemployed_MediumSkilled,27.0,0.954542,0.339835,0.335329,0.692473,0.986980,1.225957,1.516661


In [559]:
df_1.groupby('profession')['credit-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,371.0,0.985874,0.288420,0.501339,0.732561,0.981177,1.230955,1.499588
LowSkilled,279.0,1.005408,0.286638,0.503679,0.748134,1.024382,1.239014,1.486924
MediumSkilled,255.0,0.998376,0.303427,0.504979,0.733830,0.977982,1.270641,1.495749
Unemployed_HighSkilled,37.0,0.960125,0.256788,0.558812,0.724799,0.953820,1.177347,1.366257
Unemployed_LowSkilled,31.0,0.911487,0.279568,0.504190,0.680414,0.875557,1.083153,1.441907
Unemployed_MediumSkilled,27.0,0.968485,0.320866,0.521555,0.635367,0.985310,1.228268,1.471361


In [560]:
df_1.groupby('profession')['y-categorical-default'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,371.0,0.555256,0.497608,0.0,0.0,1.0,1.0,1.0
LowSkilled,279.0,0.523297,0.500354,0.0,0.0,1.0,1.0,1.0
MediumSkilled,255.0,0.415686,0.493809,0.0,0.0,0.0,1.0,1.0
Unemployed_HighSkilled,37.0,0.567568,0.502247,0.0,0.0,1.0,1.0,1.0
Unemployed_LowSkilled,31.0,0.741935,0.444803,0.0,0.5,1.0,1.0,1.0
Unemployed_MediumSkilled,27.0,0.481481,0.509175,0.0,0.0,0.0,1.0,1.0


## MODELLING PD, LGD, EAD

#### Packages

In [561]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#### Converting variables into dummies

In [562]:
# Convert categorical variables to numeric
df_R = pd.get_dummies(df_1, columns=["educational level", "profession"], drop_first=True)
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,debt-to-income ratio after credit,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled
0,name0,38,0,1,952.0,1440.368618,-488.368618,0.512992,1330.095876,0.687062,1.910152,0,False,True,False,True,False,False,False,False
1,name1,32,0,1,3092.0,2465.972725,626.027275,0.000000,1851.645292,0.535636,0.396384,1,False,False,True,False,False,False,False,False
2,name2,34,0,1,1343.0,657.893623,685.106377,0.000000,1636.794542,0.797688,0.708629,0,False,True,False,True,False,False,False,False
3,name3,60,1,0,6376.0,6100.486048,275.513952,0.000000,6392.332779,0.900734,0.959351,0,True,False,False,False,False,False,False,False
4,name4,57,0,0,2463.0,2171.529759,291.470241,0.000000,3136.363298,1.392856,1.155052,1,False,True,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,48,1,0,9316.0,6361.661732,2954.338268,0.000000,4960.434197,1.006062,0.215339,1,True,False,False,False,False,False,False,False
996,name996,40,0,0,2624.0,1089.893142,1534.106858,0.000000,3299.763186,0.852561,0.672887,1,False,False,False,False,True,False,False,False
997,name997,43,2,1,1461.0,1615.828788,-154.828788,0.105975,2022.859947,0.672195,1.490547,1,False,True,False,True,False,False,False,False
998,name998,41,1,1,1953.0,1816.165055,136.834945,0.000000,2228.939831,0.733985,1.071226,0,False,False,False,False,True,False,False,False


#### Defining X and y

In [563]:
# Column "name" is dropped from the dataframe, no need to keep it
# All the variables except y-categorical-default are X
X = df_R.drop(columns=["name", "y-categorical-default"])
y = df_R["y-categorical-default"]

#### Split between trainning and test sets

In [564]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### Standardize the variables for a better gradient descendt
Standardizing features to have a mean of zero ensures that all features are centered around the same baseline, which helps prevent models from being biased toward features with larger numerical values. It also makes gradient-based optimization methods like gradient descent behave more efficiently by ensuring all features contribute equally to the cost function.

In [565]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### LOGIT

##### Packages

In [566]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

##### Training the Logistic Regression

In [567]:
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)

LogisticRegression()

##### Predictions

In [568]:
y_pred = log_reg.predict(X_test_scaled)

##### Evaluations of the accuracy of model

In [569]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.63


In [570]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.61      0.64      0.63        97
           1       0.65      0.62      0.63       103

    accuracy                           0.63       200
   macro avg       0.63      0.63      0.63       200
weighted avg       0.63      0.63      0.63       200



##### Estimating the PDs

In [571]:
df_R["PD_LR"] = log_reg.predict_proba(scaler.transform(X))[:, 1]
df_R


,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR
0,name0,38,0,1,952.0,1440.368618,-488.368618,0.512992,1330.095876,0.687062,...,0,False,True,False,True,False,False,False,False,0.515903
1,name1,32,0,1,3092.0,2465.972725,626.027275,0.000000,1851.645292,0.535636,...,1,False,False,True,False,False,False,False,False,0.423612
2,name2,34,0,1,1343.0,657.893623,685.106377,0.000000,1636.794542,0.797688,...,0,False,True,False,True,False,False,False,False,0.342303
3,name3,60,1,0,6376.0,6100.486048,275.513952,0.000000,6392.332779,0.900734,...,0,True,False,False,False,False,False,False,False,0.533432
4,name4,57,0,0,2463.0,2171.529759,291.470241,0.000000,3136.363298,1.392856,...,1,False,True,False,True,False,False,False,False,0.520312
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,48,1,0,9316.0,6361.661732,2954.338268,0.000000,4960.434197,1.006062,...,1,True,False,False,False,False,False,False,False,0.544828
996,name996,40,0,0,2624.0,1089.893142,1534.106858,0.000000,3299.763186,0.852561,...,1,False,False,False,False,True,False,False,False,0.185523
997,name997,43,2,1,1461.0,1615.828788,-154.828788,0.105975,2022.859947,0.672195,...,1,False,True,False,True,False,False,False,False,0.661909
998,name998,41,1,1,1953.0,1816.165055,136.834945,0.000000,2228.939831,0.733985,...,0,False,False,False,False,True,False,False,False,0.296247


### Using Neuronal Networks

##### Packages

In [572]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

##### Building the Neuronal Network

In [573]:
# We may try out:

# tanh: The hyperbolic tangent function outputs values between -1 and 1, making it useful for hidden layers 
# where you want activations that are zero-centered, which helps in faster convergence and avoids saturation for small inputs.
 
# relu: The Rectified Linear Unit activation function outputs zero for any negative input and passes positive values as they are.
# It is widely used in hidden layers for its simplicity and effectiveness, and helps avoid the vanishing gradient problem seen with functions like sigmoid and tanh.

# softmax: Softmax is typically used in the output layer for multi-class classification tasks. It converts the raw outputs into probabilities, 
# ensuring that the sum of all output values equals 1, representing the probability distribution over multiple classes.

# This is like having an input which is your variable X, then 32 neurons process the input features,
# using a function (in this case "tanh") to calculate the weights and transformations at each neuron. 
# The results are then passed to a subsequent layer with 16 neurons, where again "tanh" is applied to further transform the data.
# In the end, everything is passed through a final neuron that uses a "sigmoid" (logistic) function 
# to produce an output between [0, 1], representing a probability for binary classification.
model = Sequential([  # Each layer is run after the other, forming a linear stack of layers.
    # The first Dense layer applies 32 units (neurons) and uses the "tanh" activation function.
    # The input_shape corresponds to the number of features in the dataset (X_train_scaled).
    Dense(32, activation='tanh', input_shape=(X_train_scaled.shape[1],)),
    
    # The second Dense layer applies 16 units (neurons) and uses "tanh" activation function.
    # "tanh" ensures that the output of each neuron will be between -1 and 1, centering the activations.
    Dense(16, activation='tanh'),
    
    # The final Dense layer outputs a single value, which is the probability of the positive class.
    # Sigmoid activation squashes the output to a value between 0 and 1.
    # This is commonly used for binary classification, where the output is a probability of class 1.
    Dense(1, activation='sigmoid')
])

/Users/bonjour/opt/anaconda3/envs/bankgame/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


##### Compiling the model

In [574]:
# The model is being compiled with the following parameters:
# optimizer='adam': The Adam optimizer is being used. It is an adaptive learning rate optimization algorithm that 
#  combines the benefits of both AdaGrad and RMSProp, making it well-suited for most deep learning models.
# loss='binary_crossentropy': The loss function used is binary cross-entropy, which is appropriate for binary classification 
#  tasks where the output is a probability of belonging to one of two classes, which is the case of our y-variable
# metrics=['accuracy']: The model will track accuracy as the evaluation metric during training and testing, 
#  which measures the percentage of correct predictions.

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

##### Training the model

In [575]:
# The model is fit with X_train_scaled: This means the model is being trained on the scaled training data (X_train_scaled) 
# using the corresponding labels (y_train).

# Uses 25 epochs: An epoch refers to one full pass through the entire training dataset. 
# The model will train for 25 epochs, meaning it will go through the data 25 times to learn the optimal weights.

# It will use a batch size of 32: The model will train using 32 samples (or rows of data) at a time, and after processing 
# those 32, it updates the weights before moving on to the next 32 samples. 

# During training, the model's performance is periodically evaluated on the validation set (X_test_scaled and y_test) 
# to monitor overfitting and to adjust the training accordingly.

model_NN = model.fit(X_train_scaled, y_train, epochs=25, batch_size=32, validation_data=(X_test_scaled, y_test))

Epoch 1/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4096 - loss: 0.7916 - val_accuracy: 0.5050 - val_loss: 0.7104
Epoch 2/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5438 - loss: 0.6869 - val_accuracy: 0.5150 - val_loss: 0.6854
Epoch 3/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6158 - loss: 0.6548 - val_accuracy: 0.5500 - val_loss: 0.6709
Epoch 4/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6217 - loss: 0.6492 - val_accuracy: 0.6050 - val_loss: 0.6530
Epoch 5/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6305 - loss: 0.6275 - val_accuracy: 0.6050 - val_loss: 0.6464
Epoch 6/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6433 - loss: 0.6161 - val_accuracy: 0.6250 - val_loss: 0.6375
Epoch 7/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6794 - loss: 0.6006 - val_accuracy: 0.6300 - val_loss: 0.6414
Epoch 8/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6567 - loss: 0.6061 - val_accuracy: 0.6200 - val_loss:

##### We want to get the last accuracy of the last epoch and the minimal and highest accuracy values for comparison

In [576]:
train_accuracies_NN = model_NN.history['accuracy']

In [577]:
# final accuracy value
final_accuracy = train_accuracies_NN[-1]
# maximal accuracy value
min_accuracy = min(train_accuracies_NN)
# minimal accuracy value
max_accuracy = max(train_accuracies_NN)
# average accuracy value
average_accuracy = sum(train_accuracies_NN) / len(train_accuracies_NN)

In [578]:
# Print the results
print(f'Final accuracy: {final_accuracy:.4f}')
print(f'Minimum accuracy during training of the NN: {min_accuracy:.4f}')
print(f'Maximum accuracy during training of the NN: {max_accuracy:.4f}')
print(f'Average accuracy during training of the NN: {average_accuracy:.4f}')

Final accuracy: 0.6800
Minimum accuracy during training of the NN: 0.4475
Maximum accuracy during training of the NN: 0.6800
Average accuracy during training of the NN: 0.6505


##### Estimating the PDs

In [579]:
df_R["PD_NN"]= model.predict(scaler.transform(X))
df_R

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN
0,name0,38,0,1,952.0,1440.368618,-488.368618,0.512992,1330.095876,0.687062,...,False,True,False,True,False,False,False,False,0.515903,0.580636
1,name1,32,0,1,3092.0,2465.972725,626.027275,0.000000,1851.645292,0.535636,...,False,False,True,False,False,False,False,False,0.423612,0.409486
2,name2,34,0,1,1343.0,657.893623,685.106377,0.000000,1636.794542,0.797688,...,False,True,False,True,False,False,False,False,0.342303,0.348421
3,name3,60,1,0,6376.0,6100.486048,275.513952,0.000000,6392.332779,0.900734,...,True,False,False,False,False,False,False,False,0.533432,0.497910
4,name4,57,0,0,2463.0,2171.529759,291.470241,0.000000,3136.363298,1.392856,...,False,True,False,True,False,False,False,False,0.520312,0.560181
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,48,1,0,9316.0,6361.661732,2954.338268,0.000000,4960.434197,1.006062,...,True,False,False,False,False,False,False,False,0.544828,0.624395
996,name996,40,0,0,2624.0,1089.893142,1534.106858,0.000000,3299.763186,0.852561,...,False,False,False,False,True,False,False,False,0.185523,0.187388
997,name997,43,2,1,1461.0,1615.828788,-154.828788,0.105975,2022.859947,0.672195,...,False,True,False,True,False,False,False,False,0.661909,0.701022
998,name998,41,1,1,1953.0,1816.165055,136.834945,0.000000,2228.939831,0.733985,...,False,False,False,False,True,False,False,False,0.296247,0.299329


### Using Random Forests

##### Packages

In [580]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

##### Fitting the model

In [581]:
# Initialize the RandomForestClassifier with the following parameters:
# n_estimators=100: This sets the number of decision trees (estimators) in the forest. The model will train 100 individual trees and aggregate their results to make predictions.
# random_state=42: This ensures reproducibility by fixing the random seed used in the training process. 
# To get the same result even if the code is run multiple times (obviously this will only be affected by the random nature of our data)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

##### Accuracy of the model

In [582]:
# Predict class labels for X_test_scaled
y_pred = rf_model.predict(X_test_scaled)
# Calculate accuracy by comparing the predicted labels with our simulated data y
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.5600


##### Estimating the PDs

In [584]:
# df_R = df_R.iloc[:len(X_test_scaled)]
# df_R["PD_RF"] = rf_model.predict_proba(X_test_scaled)[:, 1]
df_R["PD_RF"] = rf_model.predict_proba(scaler.transform(X))[:, 1] # needs to be checked
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN,PD_RF
0,name0,38,0,1,952.0,1440.368618,-488.368618,0.512992,1330.095876,0.687062,...,True,False,True,False,False,False,False,0.515903,0.580636,0.28
1,name1,32,0,1,3092.0,2465.972725,626.027275,0.000000,1851.645292,0.535636,...,False,True,False,False,False,False,False,0.423612,0.409486,0.78
2,name2,34,0,1,1343.0,657.893623,685.106377,0.000000,1636.794542,0.797688,...,True,False,True,False,False,False,False,0.342303,0.348421,0.61
3,name3,60,1,0,6376.0,6100.486048,275.513952,0.000000,6392.332779,0.900734,...,False,False,False,False,False,False,False,0.533432,0.497910,0.12
4,name4,57,0,0,2463.0,2171.529759,291.470241,0.000000,3136.363298,1.392856,...,True,False,True,False,False,False,False,0.520312,0.560181,0.84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,48,1,0,9316.0,6361.661732,2954.338268,0.000000,4960.434197,1.006062,...,False,False,False,False,False,False,False,0.544828,0.624395,0.78
996,name996,40,0,0,2624.0,1089.893142,1534.106858,0.000000,3299.763186,0.852561,...,False,False,False,True,False,False,False,0.185523,0.187388,0.79
997,name997,43,2,1,1461.0,1615.828788,-154.828788,0.105975,2022.859947,0.672195,...,True,False,True,False,False,False,False,0.661909,0.701022,0.84
998,name998,41,1,1,1953.0,1816.165055,136.834945,0.000000,2228.939831,0.733985,...,False,False,False,True,False,False,False,0.296247,0.299329,0.08
